# 9주차 실습 — 질의응답 · 기계번역 · 대화시스템

**자연어처리 | Chapter 9 실습 노트북**

---

### 이번 실습의 목표

질문에 답하고, 언어를 옮기고, 사용자와 대화하는 세 종류의 응용 시스템을 **직접 코드로 조립**해 보고,
각 시스템을 어떤 지표로 평가하는지를 손으로 구현하여 확인한다.

| PART | 주제 | 대응 절 | 주요 도구 |
|:---:|---|:---:|---|
| 1 | 질의응답 (Retriever + Reader, EM/F1) | 9.1 | `scikit-learn`, `transformers` |
| 2 | 기계번역 (MarianMT, 서브워드, BLEU, 빔 탐색) | 9.2 | `transformers`, `sacrebleu` |
| 3 | 대화시스템 (Frame 기반, 의도 분류, Slot Error Rate) | 9.3 | `re`, `scikit-learn`, `transformers` |

### 코드 구현의 흐름

1. **(데이터)** 문서 컬렉션 · 병렬 문장쌍 · 대화 발화 데이터를 준비한다.
2. **(모델)** 검색 모델(TF-IDF) / 독해 모델(BERT 계열) / 번역 모델(Transformer) / 분류 모델(로지스틱 회귀)을 준비한다.
3. **(파이프라인)** '이해 → 탐색·정렬 → 생성' 순서로 모듈을 연결하여 시스템을 완성한다.
4. **(평가)** EM·F1(질의응답), BLEU(번역), Slot Error Rate(대화)로 정량 평가하고 실패 사례를 분석한다.

### 진행 방법

1. 아래 **[0] 환경 설정** 셀을 먼저 실행한다. (Colab 기준 2~3분 소요, 모델 다운로드 포함)
2. 위에서부터 순서대로 셀을 실행한다.
3. 본문 중간의 **실습 TODO (1)~(8)** 은 스스로 코드를 채워야 하는 빈칸이다.
4. 노트북 맨 끝의 **[과제] 3문항**이 제출 대상이다.

> **실행 환경** : Google Colab 권장. CPU로도 동작하지만 GPU를 쓰면 번역·독해가 훨씬 빠르다.

> **[문제] 노트북 사용법**
>
> - `raise NotImplementedError(...)` 로 막혀 있는 부분이 여러분이 채워야 할 곳이다.
>   그 줄을 지우고 직접 코드를 작성한다.
> - 아직 구현하지 않은 과제가 있어도 `런타임 → 모두 실행` 은 끝까지 동작한다.
>   미구현 과제가 있는 셀은 `▶ 미구현 : ...` 이라고만 출력된다.
> - 각 과제 아래의 **채점 셀** 이 통과(`통과!` 출력)해야 정답으로 인정된다.


## [0] 환경 설정

이번 실습에서 사용하는 라이브러리는 다음과 같다.

- **transformers** — 기계독해(MRC) 파이프라인, MarianMT 번역 모델, DialoGPT 대화 모델을 불러오는 데 사용한다.
- **sentencepiece** — MarianMT 가 쓰는 서브워드 토크나이저의 백엔드다. 없으면 토크나이저 로딩이 실패한다.
- **sacrebleu** — 직접 구현한 BLEU 를 교차 검증하기 위한 표준 구현이다.
- **scikit-learn** — TF-IDF 검색기(Retriever)와 의도 분류기(로지스틱 회귀)에 사용한다. &nbsp;→ *Colab에 이미 설치되어 있다*
- **torch** — 위 모델들의 실행 백엔드다. &nbsp;→ *Colab에 이미 설치되어 있다*

> **중요** : 아래 설치 셀에서는 `transformers` 의 버전을 고정하지 않는다.
> 예전 안내처럼 `transformers` 만 특정 버전으로 고정하면, 함께 쓰이는 `tokenizers` · `huggingface_hub` 는
> 최신 버전 그대로 남아 서로 맞지 않는 조합이 만들어지고
> `ImportError: cannot import name ... from huggingface_hub` 같은 오류가 난다.
> 세 패키지는 **함께 고정하거나 함께 풀어야** 하며, 여기서는 모두 풀어 최신 호환 조합을 그대로 쓴다.

> **설치 후 재시작 안내가 뜨면** : `RESTART SESSION`(런타임 다시 시작)을 눌러 **반드시 재시작**한다.
> 재시작하면 지금까지 실행한 변수는 사라지지만 설치된 패키지는 그대로 남는다.
> 재시작 뒤에는 **바로 다음 셀([0-2] 공통 import)부터 이어서** 실행하면 된다.
> (설치 셀을 다시 실행할 필요는 없다.) 재시작하지 않고 진행하면 이전 버전이 메모리에 남아 있어
> 모델 로딩 단계에서 알 수 없는 오류가 발생할 수 있다.

아래 설치 셀은 세션당 한 번만 실행하면 된다.


In [ ]:
# ── [0-1] 라이브러리 설치 (Colab 기준, 약 1~2분) ──────────────────
# 버전을 고정하지 않는다. transformers 만 고정하면 tokenizers / huggingface_hub 와
# 버전이 어긋나 import 단계에서 깨진다. (세 패키지는 함께 고정하거나 함께 풀어야 한다)
!pip install -q transformers sentencepiece sacrebleu

# scikit-learn 과 torch 는 Colab 에 이미 들어 있으므로 다시 설치하지 않는다.
# (재설치하면 numpy 등 의존 패키지가 교체되며 런타임 재시작이 강제된다)

# 그래프의 한글이 깨지지 않도록 나눔 폰트 설치
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

print("설치 완료 — 재시작 안내가 뜨면 런타임을 다시 시작한 뒤 [0-2] 셀부터 이어서 실행한다.")


In [ ]:
# ── [0-2] 공통 import 및 한글 폰트 설정 ───────────────────────────
# (설치 셀에서 런타임을 재시작했다면 이 셀부터 다시 실행하면 된다)
import re
import math
import warnings
from collections import Counter

import numpy as np

import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")   # 경고 메시지 숨김 (결과 확인을 쉽게 하기 위함)

# 한글 폰트 설정 ---------------------------------------------------------
# 주의 : apt 로 폰트를 설치해도 matplotlib 의 폰트 캐시는 갱신되지 않기 때문에
#        plt.rcParams['font.family'] 만 바꾸면 그래프의 한글이 그대로 깨진다.
#        addfont() 로 폰트 파일을 직접 등록해야 재시작 없이 즉시 적용된다.
import os
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

NANUM = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if os.path.exists(NANUM):
    fm.fontManager.addfont(NANUM)
    plt.rcParams['font.family'] = 'NanumGothic'
else:
    for cand in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'Noto Sans CJK KR']:
        if any(f.name == cand for f in fm.fontManager.ttflist):
            plt.rcParams['font.family'] = cand
            break
plt.rcParams['axes.unicode_minus'] = False

print("torch:", torch.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())
print("matplotlib 한글 폰트:", plt.rcParams['font.family'])

DEVICE = 0 if torch.cuda.is_available() else -1   # transformers pipeline 용 device 인자

# 주피터/코랩이 아닌 환경에서도 display() 가 동작하도록 대비
try:
    display
except NameError:
    def display(x):
        print(x)


---
# Part 1. 질의응답 (Question Answering)

강의자료 9.1절의 **정보 검색 기반 질의응답** 구조를 코드로 구현한다.

| 강의자료의 단계 | 이 실습에서의 구현 |
|---|---|
| 질문 처리 | 질문 토큰화 · 질의 벡터 생성 |
| 문서 처리 | TF-IDF Retriever 로 관련 문서 검색 |
| 정답 처리 | 사전학습 MRC Reader(KoELECTRA)로 정답 구간 추출 · 점수로 순위화 |

## 1-1. 문서 컬렉션 준비

- 실제 시스템은 위키피디아 전체를 색인하지만, 여기서는 이해를 위해 작은 문서 집합을 사용한다.

In [ ]:
# QA 시스템이 검색할 문서 컬렉션을 정의한다. Retriever가 이 안에서 근거 문서를 고르고, Reader가 그 안에서 정답 구간을 찾는다.
# 각 문서는 두 문장으로 되어 있고 인물 · 역사 · 모델 · 평가지표 · 대화 시스템 등 주제를 섞어 두었다.
# 질문에 따라 서로 다른 문서가 검색되는지 확인하기 위한 구성이다.
DOCS = [
    "마하트마 간디는 1869년 10월 2일 인도 구자라트주 포르반다르에서 태어났다. "
    "그는 인도의 독립 운동을 이끈 지도자로, 비폭력 저항 운동인 사티아그라하를 주장하였다.",

    "1930년 간디는 영국의 소금세에 저항하기 위해 소금 행진을 시작하였다. "
    "이 행진은 아메다바드에서 단디까지 약 390킬로미터에 걸쳐 진행되었다.",

    "자와할랄 네루는 인도의 초대 총리로 1947년에 취임하였다. "
    "그는 간디의 동료이자 인도 국민회의의 핵심 인물이었다.",

    "세종대왕은 1443년에 훈민정음을 창제하였고 1446년에 반포하였다. "
    "훈민정음은 백성이 쉽게 글을 배울 수 있도록 만든 문자이다.",

    "트랜스포머는 2017년에 제안된 신경망 구조로, 셀프 어텐션만으로 문맥을 표현한다. "
    "순차 계산을 제거하여 병렬 학습이 가능하며 기계번역 성능을 크게 향상시켰다.",

    "BLEU는 기계 번역문과 참조 번역문의 n-gram 정밀도를 기하평균한 평가 지표이다. "
    "짧은 번역에 벌점을 주기 위해 길이 벌점(brevity penalty)을 곱한다.",

    "대화 시스템은 음성 인식, 자연어 이해, 대화 관리, 자연어 생성, 음성 합성의 다섯 모듈로 구성된다. "
    "자연어 이해 단계에서는 의도를 분류하고 슬롯을 채운다.",
]

print("문서 수:", len(DOCS))
for i, d in enumerate(DOCS):
    print(f"[{i}] {d[:40]}...")

## 1-2. 문서 처리 단계 — TF-IDF Retriever

- 강의자료의 수식을 그대로 사용한다.

$$ \text{tf-idf}(t, d) = tf(t,d) \times \log \frac{N}{df(t)}, \qquad sim(q,d) = \cos(v_q, v_d) $$

- 한국어는 조사가 붙어 표면형이 달라지므로, 여기서는 단어 대신 **문자 n-gram만**을 자질로 사용해 어휘 불일치를 완화한다.
- 참고 : scikit-learn의 `TfidfVectorizer` 는 기본값으로 스무딩된 IDF `log((1+N)/(1+df))+1` 과 L2 정규화를 적용하므로, 값 자체는 위 식과 조금 다르다.

In [ ]:
# 단어 경계를 유지한 문자 2~3-gram 만 사용하는 벡터라이저 (조사 변화에 강하다)
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))
DOC_MATRIX = vectorizer.fit_transform(DOCS)

print("문서-단어 행렬 크기:", DOC_MATRIX.shape)   # (문서 수, 자질 수)

In [ ]:
def retrieve(question, top_k=2):
    """질문과 가장 유사한 문서 top_k개를 (문서번호, 유사도) 리스트로 반환한다."""
    q_vec = vectorizer.transform([question])

    # TODO (1) : q_vec 과 DOC_MATRIX 의 코사인 유사도를 계산하여 sims 에 저장한다.
    #   - cosine_similarity(q_vec, DOC_MATRIX) 의 결과는 (1, 문서수) 모양의 **2차원** 배열이다.
    #   - 아래 argsort 와 sims[i] 인덱싱이 제대로 동작하려면
    #     sims 는 반드시 (문서수,) 모양의 **1차원** 배열이어야 한다.
    #   - 따라서 2차원 결과를 그대로 넣지 말고, 첫 번째 행만 꺼내거나 평탄화하여 1차원으로 만들 것.
    #     (2차원 그대로 두면 `for i in order` 가 배열을 순회하게 되어 int(i) 에서 오류가 난다)
    raise NotImplementedError('실습 TODO (1) : 질문 벡터와 DOC_MATRIX 의 코사인 유사도를 1차원 배열 sims 로 계산하시오.')

    order = np.argsort(-sims)[:top_k]          # 유사도 내림차순 정렬 후 상위 k개
    return [(int(i), float(sims[i])) for i in order]


# 확인
try:
    q = "인도의 독립 운동을 이끈 사람은 누구인가?"
    for idx, score in retrieve(q, top_k=3):
        print(f"score={score:.3f}  [{idx}] {DOCS[idx][:45]}...")
except NotImplementedError as e:
    print('▶ 미구현 :', e)


> **점검** : 0번 문서가 1위로 나와야 한다.
>
> - `sims` 가 **1차원 배열**((문서수,) 모양)인지 먼저 확인하자. 2차원 (1, 문서수) 그대로 두면
>   `np.argsort` 결과도 2차원이 되어 `int(i)` 에서 `TypeError` 가 난다.
> - 유사도 점수가 모두 0이면 `sims` 계산 자체가 잘못된 것이다.


## 1-3. 정답 처리 단계 — 기계독해 Reader

- 지문과 질문을 함께 입력하여 정답 구간(span)의 시작·끝 위치를 예측하는 모델을 사용한다.
- 한국어 KorQuAD로 미세조정된 공개 모델을 불러온다. (최초 실행 시 모델 다운로드에 시간이 걸린다)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

# 한국어 기계독해(MRC) 모델
MRC_MODEL = "monologg/koelectra-small-v2-distilled-korquad-384"

tokenizer = AutoTokenizer.from_pretrained(MRC_MODEL)
model = AutoModelForQuestionAnswering.from_pretrained(MRC_MODEL)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()


def reader(question, context):
    inputs = tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation="only_second",
        max_length=384,
        return_offsets_mapping=True
    )

    offsets = inputs.pop("offset_mapping")[0]
    sequence_ids = inputs.sequence_ids(0)

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    start_prob = torch.softmax(outputs.start_logits[0], dim=-1)
    end_prob = torch.softmax(outputs.end_logits[0], dim=-1)

    # context에 해당하는 토큰만 후보로 사용
    context_idx = [
        i for i, sid in enumerate(sequence_ids)
        if sid == 1
    ]

    best_score = -1
    best_start = None
    best_end = None

    # 가장 점수가 높은 answer span 탐색
    for i in context_idx:
        for j in context_idx:
            if j < i or j - i > 30:
                continue

            score = (start_prob[i] * end_prob[j]).item()

            if score > best_score:
                best_score = score
                best_start = i
                best_end = j

    start_char = offsets[best_start][0].item()
    end_char = offsets[best_end][1].item()

    return {
        "score": best_score,
        "start": start_char,
        "end": end_char,
        "answer": context[start_char:end_char]
    }


out = reader(
    question="간디는 언제 태어났는가?",
    context=DOCS[0]
)

print(out)      # {'score':..., 'start':..., 'end':..., 'answer': '1869년 10월 2일'}

In [ ]:
# 정답 유형별로 잘 되는 질문 / 어려운 질문을 비교해 보자
tests = [
    ("간디는 언제 태어났는가?", DOCS[0]),
    ("소금 행진의 거리는 얼마인가?", DOCS[1]),
    ("인도의 초대 총리는 누구인가?", DOCS[2]),
    ("훈민정음은 언제 반포되었는가?", DOCS[3]),
    ("간디는 왜 비폭력을 주장했는가?", DOCS[0]),   # 서술형(non-factoid) → 어려움
]
for q, c in tests:
    r = reader(question=q, context=c)
    print(f"Q: {q}\n  → A: {r['answer']}  (score={r['score']:.3f})")

> **관찰 포인트**: 마지막 질문처럼 정답이 지문에 문자열로 존재하지 않는 서술형 질문에서는
> 점수가 낮거나 엉뚱한 구간을 반환한다. 이것이 추출형(extractive) QA의 구조적 한계다.

## 1-4. Retriever + Reader 연결 (미니 오픈도메인 QA)

- 이제 문서를 지정하지 않고 질문만 주어도 답할 수 있는 파이프라인을 만든다.

In [ ]:
def open_domain_qa(question, top_k=3, verbose=True):
    """검색된 top_k 문서에 대해 Reader를 실행하고, 가장 점수가 높은 정답을 반환한다."""
    candidates = []
    for doc_id, sim in retrieve(question, top_k=top_k):
        r = reader(question=question, context=DOCS[doc_id])

        # TODO (2) : 최종 점수(final_score)를 만든다.
        #   - 검색 유사도(sim)와 Reader 점수(r["score"])를 **함께** 반영해야 한다.
        #   - 두 값의 가중합(가중치는 스스로 정한다)이나 곱 중 무엇을 써도 좋다.
        #   - 어느 쪽이든 두 값이 모두 커질 때 final_score 도 커져야 한다.
        #   - 어떤 방식을 골랐고 왜 그렇게 정했는지는 과제 1 서술형 답안에서 근거로 쓸 수 있다.
        raise NotImplementedError('실습 TODO (2) : 검색 유사도와 Reader 점수를 함께 반영한 final_score 를 계산하시오.')

        candidates.append({"doc_id": doc_id, "sim": sim,
                           "answer": r["answer"], "reader": r["score"],
                           "final": final_score})

    candidates.sort(key=lambda c: c["final"], reverse=True)
    if verbose:
        for c in candidates:
            print(f"  [doc {c['doc_id']}] sim={c['sim']:.3f} reader={c['reader']:.3f} "
                  f"final={c['final']:.3f} → {c['answer']}")
    best = candidates[0]
    return best["answer"], best


try:
    for q in ["인도의 독립 운동을 이끈 사람은 누구인가?",
              "훈민정음을 창제한 사람은 누구인가?",
              "트랜스포머는 언제 제안되었는가?"]:
        print("Q:", q)
        ans, _ = open_domain_qa(q)
        print("  ==> 최종 정답:", ans, "\n")
except NotImplementedError as e:
    print('▶ 미구현 :', e)


## 1-5. 질의응답의 평가 — EM과 F1

$$ EM = \frac{\text{완전히 일치한 질문 수}}{\text{전체 질문 수}}, \qquad
F1 = \frac{2 \cdot precision \cdot recall}{precision + recall} $$

- F1은 정답과 예측을 **토큰 단위**로 비교하여 겹치는 정도를 계산한다.

In [ ]:
def normalize(text):
    """공백·문장부호를 제거하고 소문자화하는 표준화 함수"""
    text = re.sub(r"[^0-9A-Za-z가-힣 ]", " ", text)
    return " ".join(text.lower().split())


def exact_match(pred, gold):
    return int(normalize(pred) == normalize(gold))


def token_f1(pred, gold):
    pred_tokens = normalize(pred).split()
    gold_tokens = normalize(gold).split()
    if not pred_tokens or not gold_tokens:
        return 0.0

    common = Counter(pred_tokens) & Counter(gold_tokens)
    n_same = sum(common.values())        # 겹치는 토큰 개수
    if n_same == 0:
        return 0.0

    # TODO (3) : precision, recall, f1 을 계산하여 f1 을 반환한다.
    #   - precision = 겹치는 토큰 수(n_same) / 예측 토큰 수
    #   - recall    = 겹치는 토큰 수(n_same) / 정답 토큰 수
    #   - f1        = 두 값의 조화평균
    raise NotImplementedError('실습 TODO (3) : precision·recall 로부터 토큰 단위 F1 을 계산해 반환하시오.')

    return f1


# 강의자료의 예시로 검증 (기대값: 1.00 / 0.67 / 0.00)
try:
    for pred, gold in [("마하트마 간디", "마하트마 간디"),
                       ("간디", "마하트마 간디"),
                       ("나렌드라 모디", "마하트마 간디")]:
        print(f"pred={pred:12s} gold={gold:12s} EM={exact_match(pred, gold)} F1={token_f1(pred, gold):.2f}")
except NotImplementedError as e:
    print('▶ 미구현 :', e)


In [ ]:
# 소규모 평가셋으로 시스템 전체를 평가한다
EVAL_SET = [
    ("인도의 독립 운동을 이끈 사람은 누구인가?", "마하트마 간디"),
    ("간디는 언제 태어났는가?", "1869년 10월 2일"),
    ("소금 행진은 어디에서 시작되었는가?", "아메다바드"),
    ("인도의 초대 총리는 누구인가?", "자와할랄 네루"),
    ("훈민정음은 언제 창제되었는가?", "1443년"),
    ("트랜스포머는 언제 제안되었는가?", "2017년"),
]

em_scores, f1_scores = [], []
try:
    for question, gold in EVAL_SET:
        pred, _ = open_domain_qa(question, top_k=3, verbose=False)
        em_scores.append(exact_match(pred, gold))
        f1_scores.append(token_f1(pred, gold))
        print(f"Q: {question}\n   예측: {pred}  |  정답: {gold}  |  EM={em_scores[-1]} F1={f1_scores[-1]:.2f}")

    print(f"\n전체 EM = {100*np.mean(em_scores):.1f}   전체 F1 = {100*np.mean(f1_scores):.1f}")
except NotImplementedError as e:
    em_scores, f1_scores = [], []          # 미구현 시에도 뒤 셀이 참조할 수 있도록 초기화
    print('▶ 미구현 :', e)


> **토의**: EM은 0인데 F1은 높은 사례를 찾아보자. 왜 이런 현상이 생기는가?
> (예: 정답 "1869년 10월 2일" 을 "1869년" 으로만 예측한 경우)

---
# Part 2. 기계번역 (Machine Translation)

강의자료 9.2절의 통계 기반 번역(BLEU, 정렬)과 신경망 번역(Transformer, 빔 탐색)을 실습한다.

## 2-1. 사전학습 번역 모델로 번역하기

- Helsinki-NLP의 OPUS-MT 모델(Transformer 기반 인코더–디코더)을 사용한다.

In [ ]:
# 사전학습된 Transformer 인코더-디코더(MarianMT)로 한국어를 영어로 번역한다.
from transformers import MarianMTModel, MarianTokenizer

# OPUS 병렬 코퍼스로 학습된 한국어 → 영어 번역 모델
KO2EN = "Helsinki-NLP/opus-mt-ko-en"
tok_ko2en = MarianTokenizer.from_pretrained(KO2EN)
model_ko2en = MarianMTModel.from_pretrained(KO2EN)
if torch.cuda.is_available():
    model_ko2en = model_ko2en.cuda()   # GPU가 있으면 올려 두어 생성 속도를 높인다

def translate(texts, tokenizer, model, num_beams=4, max_new_tokens=64):
    """문장 리스트를 번역해 리스트로 반환한다."""
    if isinstance(texts, str):
        texts = [texts]   # 문자열 하나만 들어와도 배치로 처리할 수 있게 리스트로 감싼다
    # 길이가 서로 다른 문장을 한 배치로 묶기 위해 padding=True 로 길이를 맞춘다.
    batch = tokenizer(texts, return_tensors="pt", padding=True)
    if torch.cuda.is_available():
        batch = {k: v.cuda() for k, v in batch.items()}
    with torch.no_grad():   # 추론만 하므로 기울기 계산을 꺼 메모리와 시간을 아낀다
        # num_beams > 1 이면 빔 탐색, 1 이면 그리디 탐색이 된다. max_new_tokens 는 생성 길이의 상한이다.
        gen = model.generate(**batch, num_beams=num_beams, max_new_tokens=max_new_tokens)
    # 생성된 토큰 id 열을 다시 문자열로 되돌린다. <pad>, </s> 같은 특수 토큰은 제외한다.
    return tokenizer.batch_decode(gen, skip_special_tokens=True)


# 한국어(SOV)와 영어(SVO)의 어순이 어떻게 뒤바뀌는지 관찰하기 위한 예문
samples = [
    "나는 어제 도서관에서 자연어처리 책을 읽었다.",
    "내일 나는 프랑스에서 열리는 컨퍼런스에 비행기로 간다.",
    "이 문장은 기계 번역이 어순을 어떻게 처리하는지 보여준다.",
]
for src, hyp in zip(samples, translate(samples, tok_ko2en, model_ko2en)):
    print("KO:", src)
    print("EN:", hyp, "\n")

## 2-2. 서브워드 토큰화 관찰

- 강의자료에서 다룬 **BPE/SentencePiece** 서브워드 분절이 실제로 어떻게 나뉘는지 확인한다.
- 한국어는 교착어이므로 하나의 어절이 여러 조각으로 나뉜다.

In [ ]:
# 번역 모델이 사용하는 서브워드(SentencePiece) 분절 결과를 직접 눈으로 확인한다.
for text in ["자연어처리", "먹었더라면", "컨퍼런스에", "비폭력저항운동"]:
    # 사전에 없는 긴 어절일수록 여러 조각으로 잘게 나뉜다. 미등록어(OOV)는 사라지지만 입력 길이는 길어진다.
    pieces = tok_ko2en.tokenize(text)
    ids = tok_ko2en.convert_tokens_to_ids(pieces)   # 실제 모델에 들어가는 것은 이 정수 id 열이다
    print(f"{text}  →  {pieces}  {ids}")

# 서브워드 방식은 수만 개 규모의 작은 사전만으로도 어떤 문자열이든 표현할 수 있다.
print("\n어휘 사전 크기:", tok_ko2en.vocab_size)

> **관찰 포인트**: 사전에 없는 단어(미등록어, OOV)도 서브워드 조각의 조합으로 표현되므로
> `<unk>` 로 버려지지 않는다. 이것이 서브워드 분절을 쓰는 이유다.

## 2-3. BLEU 직접 구현하기

$$ BLEU = BP \cdot \exp\Big(\sum_{n=1}^{N} w_n \log p_n\Big), \qquad
BP = \begin{cases} 1 & (c > r) \\ e^{1 - r/c} & (c \le r) \end{cases} $$

- $p_n$ : n-gram 정밀도 (같은 n-gram을 중복해서 세지 않도록 **clipping** 필요)
- $c$ : 후보 번역 길이, $r$ : 참조 번역 길이

In [ ]:
def ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]


def modified_precision(cand_tokens, ref_tokens, n):
    """clipping 을 적용한 n-gram 정밀도 p_n 을 반환한다."""
    cand_counts = Counter(ngrams(cand_tokens, n))
    ref_counts = Counter(ngrams(ref_tokens, n))
    if not cand_counts:
        return 0.0

    # TODO (4) : clipping 을 적용한 일치 개수(clipped)를 계산한다.
    #   - 후보에 등장한 각 n-gram 마다 min(후보 등장 횟수, 참조 등장 횟수) 를 더한다.
    #   - 참조에 없는 n-gram 의 참조 등장 횟수는 0 이다.
    raise NotImplementedError('실습 TODO (4) : clipping 을 적용한 일치 n-gram 개수 clipped 를 계산하시오.')

    total = sum(cand_counts.values())
    return clipped / total


def my_bleu(candidate, reference, N=4):
    cand_tokens = candidate.lower().split()
    ref_tokens = reference.lower().split()
    c, r = len(cand_tokens), len(ref_tokens)
    if c == 0:
        # 정상 경로와 같은 3-튜플을 돌려주어야 호출부의 [0] 인덱싱이 깨지지 않는다.
        return 0.0, [0.0] * N, 0.0

    precisions = [modified_precision(cand_tokens, ref_tokens, n) for n in range(1, N + 1)]
    if min(precisions) == 0:
        # 0인 정밀도가 있으면 로그가 정의되지 않으므로 아주 작은 값으로 대체(스무딩)
        precisions = [p if p > 0 else 1e-9 for p in precisions]

    log_mean = sum((1 / N) * math.log(p) for p in precisions)

    # TODO (5) : 길이 벌점 BP 를 계산한다. (c > r 이면 1, 그렇지 않으면 exp(1 - r/c))
    raise NotImplementedError('실습 TODO (5) : 길이 벌점 BP 를 계산하시오. (c > r 이면 1, 아니면 exp(1 - r/c))')

    return bp * math.exp(log_mean), precisions, bp


# 강의자료의 예시로 검증 (기대: p1=1.000, p2=0.750, p3=0.667, p4=0.500, BP≈0.819, BLEU≈0.58)
ref = "the cat is on the mat"
cand = "the cat is on mat"
try:
    score, ps, bp = my_bleu(cand, ref)
    print("정밀도:", [f"{p:.3f}" for p in ps])
    print(f"BP = {bp:.3f}")
    print(f"BLEU = {score:.4f}")
except NotImplementedError as e:
    print('▶ 미구현 :', e)


In [ ]:
# sacrebleu 와 비교 (구현이 맞았는지 교차 검증)
import sacrebleu

try:
    print("내 구현 :", f"{my_bleu(cand, ref)[0]*100:.2f}")
    print("sacrebleu:", f"{sacrebleu.sentence_bleu(cand, [ref]).score:.2f}")
    print("\n※ 스무딩 방식이 달라 소수점 차이는 정상이다. 경향이 같으면 성공.")
except NotImplementedError as e:
    print('▶ 미구현 :', e)


## 2-4. 그리디 탐색 vs 빔 탐색

- 강의자료의 '디코딩' 부분에 해당한다. 후보를 몇 개 유지하며 탐색하는지에 따라 결과가 달라진다.

In [ ]:
# 빔 크기에 따라 번역 결과와 BLEU 가 어떻게 달라지는지 비교한다. num_beams=1 이 곧 그리디 탐색이다.
src = "내일 나는 프랑스에서 열리는 컨퍼런스에 비행기로 간다."
reference_en = "tomorrow i will fly to the conference in france"   # BLEU 계산의 기준이 되는 참조 번역

for beams in [1, 2, 4, 8]:
    hyp = translate(src, tok_ko2en, model_ko2en, num_beams=beams)[0]
    try:
        # 대소문자 차이로 n-gram 이 어긋나지 않도록 소문자로 맞춘 뒤,
        # 0~1 범위의 값에 100을 곱해 일반적인 BLEU 점수 스케일로 바꾼다.
        bleu = my_bleu(hyp.lower(), reference_en)[0] * 100
        print(f"num_beams={beams}: BLEU={bleu:5.2f} | {hyp}")
    except NotImplementedError as e:
        print(f"num_beams={beams}: BLEU=  ??? | {hyp}")
        print('▶ 미구현 :', e)
        break


> **관찰 포인트**: 빔 크기를 키우면 보통 점수가 올라가지만 무한히 좋아지지는 않고,
> 계산량은 빔 크기에 비례해 늘어난다. 또한 빔이 커지면 짧고 무난한 번역을 선호하는 경향이 있다.

## 2-5. 역번역 (back-translation)

- 한국어 → 영어 → 한국어로 되돌린 뒤 원문과 비교한다.
- 데이터 증강 기법이면서, 번역 품질을 간접적으로 진단하는 방법으로도 쓰인다.

In [ ]:
EN2KO = "Helsinki-NLP/opus-mt-tc-big-en-ko"   # 영어→한국어 모델 (opus-mt-en-ko 는 존재하지 않는다)
tok_en2ko = MarianTokenizer.from_pretrained(EN2KO)
model_en2ko = MarianMTModel.from_pretrained(EN2KO)
if torch.cuda.is_available():
    model_en2ko = model_en2ko.cuda()

originals = [
    "나는 어제 도서관에서 자연어처리 책을 읽었다.",
    "대화 시스템은 사용자의 의도를 파악하고 슬롯을 채운다.",
    "기계 번역의 품질은 코퍼스의 크기에 크게 의존한다.",
]

mids = translate(originals, tok_ko2en, model_ko2en)      # ko -> en
backs = translate(mids, tok_en2ko, model_en2ko)          # en -> ko


def round_trip_bleu(original, back_translated):
    """원문(참조)과 역번역문(후보) 사이의 BLEU 를 0~100 스케일로 반환한다."""
    # TODO (6) : my_bleu 를 이용해 BLEU 를 구한 뒤 100 을 곱해 반환한다.
    #   - my_bleu(후보, 참조) 순서이며, 반환값의 첫 원소가 0~1 범위의 BLEU 다.
    #   - 여기서는 역번역문이 후보, 원문이 참조다.
    raise NotImplementedError('실습 TODO (6) : 원문과 역번역문 사이의 round-trip BLEU 를 0~100 스케일로 반환하시오.')


try:
    for o, m, b in zip(originals, mids, backs):
        print("원문   :", o)
        print("중간(EN):", m)
        print("역번역 :", b)
        print(f"round-trip BLEU = {round_trip_bleu(o, b):.2f}\n")
except NotImplementedError as e:
    print('▶ 미구현 :', e)


> **주의**: round-trip BLEU가 낮다고 반드시 번역이 나쁜 것은 아니다.
> 같은 의미를 다른 표현으로 옮기면 BLEU는 떨어진다 — 강의자료에서 지적한 BLEU의 한계다.

---
# Part 3. 대화시스템 (Dialogue System)

강의자료 9.3절의 **Frame 기반 대화 관리**, **의도 분류 + 슬롯 채우기**, **Slot Error Rate**를 구현한다.

## 3-1. Frame 기반 규칙 챗봇 (슬롯 채우기)

- Frame: 채워야 할 슬롯 목록과 우선순위, 그리고 슬롯별 질문 문장으로 구성된다.
- 빈 슬롯이 모두 채워지면 작업 관리 시스템(예약 API)을 호출한다.

In [ ]:
FRAME = {
    "movie":  {"질문": "어떤 영화를 예매하시겠습니까?", "값": None, "우선순위": 1},
    "date":   {"질문": "며칠에 관람하시겠습니까?",       "값": None, "우선순위": 2},
    "time":   {"질문": "몇 시 상영을 원하십니까?",       "값": None, "우선순위": 3},
    "count":  {"질문": "몇 분이 관람하시겠습니까?",       "값": None, "우선순위": 4},
}

MOVIES = ["아바타", "인터스텔라", "기생충", "라라랜드"]


def extract_slots(utterance):
    """발화에서 슬롯 값을 추출하여 딕셔너리로 반환한다."""
    slots = {}

    # 영화 제목: 사전 매칭
    for m in MOVIES:
        if m in utterance:
            slots["movie"] = m

    # 날짜: '토요일', '3월 5일', '내일' 등
    m = re.search(r"(월요일|화요일|수요일|목요일|금요일|토요일|일요일|오늘|내일|모레|\d+월\s*\d+일)", utterance)
    if m:
        slots["date"] = m.group(1)

    # TODO (7) : 시간(time) 슬롯을 추출하는 정규식을 작성한다.
    #   - 인식해야 하는 예 : "7시", "저녁 7시", "19시 30분", "오후 3시"
    #   - 정규식이 갖추어야 할 조건
    #       (a) '오전 / 오후 / 저녁 / 아침 / 밤' 같은 시간대 표현이 앞에 올 수도, 없을 수도 있다(선택).
    #       (b) 숫자 뒤에 '시' 가 오는 부분은 반드시 포함한다.
    #       (c) 그 뒤에 숫자 + '분' 이 이어질 수도 있다(선택).
    #       (d) (a)~(c) 전체를 하나의 괄호로 묶어 m.group(1) 로 한 번에 꺼낼 수 있어야 한다.
    #       (e) 시간대 표현과 숫자 사이의 공백은 있을 수도 없을 수도 있다.
    #   - "저녁 7시" 처럼 시간대 표현까지 통째로 잡혀야 하며, "7시" 만 잡히면 안 된다.
    raise NotImplementedError('실습 TODO (7) : 시간 슬롯을 추출하는 정규식을 작성해 m 에 re.search 결과를 담으시오.')

    if m:
        slots["time"] = m.group(1).strip()

    # 인원: '2명', '두 장', '3장'
    m = re.search(r"(\d+)\s*(명|장|인)", utterance)
    if m:
        slots["count"] = m.group(1) + "명"
    return slots


try:
    print(extract_slots("토요일 저녁 7시에 아바타 2명 예매해줘"))
    # 기대: {'movie': '아바타', 'date': '토요일', 'time': '저녁 7시', 'count': '2명'}
except NotImplementedError as e:
    print('▶ 미구현 :', e)


In [ ]:
def frame_chatbot(utterances):
    """사용자 발화 목록을 순서대로 처리하는 Frame 기반 대화 관리자"""
    frame = {k: dict(v) for k, v in FRAME.items()}     # 프레임 초기화(복사)

    for utt in utterances:
        print("사용자:", utt)
        found = extract_slots(utt)
        for k, v in found.items():
            frame[k]["값"] = v                          # 슬롯 갱신(재언급 시 값 변경)

        empty = [k for k, v in sorted(frame.items(), key=lambda x: x[1]["우선순위"])
                 if v["값"] is None]
        if empty:
            print("시스템:", frame[empty[0]]["질문"])
        else:
            summary = ", ".join(f"{k}={v['값']}" for k, v in frame.items())
            print(f"시스템: 예약을 확정합니다. ({summary})")
            print("        [작업 관리 시스템 호출: booking_api(...)]")
            break
    return frame


try:
    final_frame = frame_chatbot([
        "영화 예매하고 싶어요",
        "아바타 보려고요",
        "토요일에요",
        "저녁 7시요",
        "2명이요",
    ])
except NotImplementedError as e:
    final_frame = {}
    print('▶ 미구현 :', e)


> **관찰 포인트**: 사용자가 순서를 지키지 않고 여러 정보를 한 번에 말해도 처리된다(mixed initiative).
> 반면 사전에 없는 영화 제목이나 예상하지 못한 표현은 전혀 인식하지 못한다 — 규칙 기반의 한계다.

## 3-2. 의도(intent) 분류기 학습

- 강의자료의 자연어 이해(NLU) 모듈 중 **의도 분류**를 문장 분류 문제로 구현한다.

In [ ]:
# 대화 시스템의 자연어 이해(NLU) 모듈 가운데 의도 분류를 TF-IDF + 로지스틱 회귀로 구현한다.

# 4개 의도(영화 예매 · 날씨 · 맛집 · 인사)에 대해 의도당 4문장씩 준비한 학습 데이터
TRAIN = [
    ("영화 예매하고 싶어요", "movie_booking"),
    ("아바타 표 두 장 예매해줘", "movie_booking"),
    ("주말 영화 자리 있나요", "movie_booking"),
    ("영화 티켓 사려고요", "movie_booking"),
    ("오늘 서울 날씨 알려줘", "weather"),
    ("내일 비 오나요", "weather"),
    ("부산 기온이 몇 도야", "weather"),
    ("주말에 날씨 어때", "weather"),
    ("근처 맛집 추천해줘", "restaurant"),
    ("이탈리안 식당 알려줘", "restaurant"),
    ("저녁 먹을 곳 찾아줘", "restaurant"),
    ("파스타 맛있는 데 어디야", "restaurant"),
    ("안녕", "greeting"),
    ("반가워요", "greeting"),
    ("고마워", "greeting"),
    ("잘 있었어?", "greeting"),
]

# 학습에 없던 표현으로 만든 평가 데이터. 문장을 외운 것이 아니라 일반화가 되는지 확인한다.
TEST = [
    ("기생충 표 예매할래", "movie_booking"),
    ("대구 날씨 어때", "weather"),
    ("중국집 추천 좀", "restaurant"),
    ("안녕하세요", "greeting"),
]

X_text = [t for t, _ in TRAIN]
y = [l for _, l in TRAIN]

# 조사 · 어미 변화에 강하도록 어절 경계를 지키는 문자 2~3-gram 을 자질로 쓴다.
# ("예매해줘"와 "예매할래"가 "예매"라는 조각을 공유하게 된다)
intent_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))
X = intent_vec.fit_transform(X_text)

# 학습 문장이 16개뿐이라 선형 분류기로 충분하다. max_iter 는 수렴 경고가 뜨지 않도록 넉넉히 준다.
clf = LogisticRegression(max_iter=1000).fit(X, y)

correct = 0
for text, gold in TEST:
    # 평가 문장은 학습 때 만든 벡터라이저로 transform 만 해야 한다. (다시 fit 하면 자질 공간이 어긋난다)
    pred = clf.predict(intent_vec.transform([text]))[0]
    prob = clf.predict_proba(intent_vec.transform([text])).max()   # 최대 확률은 신뢰도로 쓸 수 있다 (낮으면 되묻기로 처리)
    correct += (pred == gold)
    print(f"{text:20s} → {pred:15s} (p={prob:.2f})  정답={gold}")

print(f"\n의도 분류 정확도: {correct}/{len(TEST)}")

## 3-3. 대화 시스템의 평가 — Slot Error Rate

$$ \text{Slot Error Rate} = \frac{\text{틀린 슬롯 수}}{\text{전체 슬롯 수}} $$

In [ ]:
def slot_error_rate(gold_slots, pred_slots):
    """정답 슬롯 딕셔너리와 예측 슬롯 딕셔너리를 비교해 SER 을 반환한다."""
    keys = set(gold_slots) | set(pred_slots)

    # TODO (8) : 틀린 슬롯 수(errors)를 계산한다.
    #   - keys 의 각 슬롯에 대해, 정답과 예측의 값이 다르면 오류 1개로 센다.
    #   - 한쪽에만 존재하는 슬롯(누락 / 잘못 추가)도 값이 다른 것으로 보아 오류로 센다.
    #   - 힌트 : 없는 키를 안전하게 읽으려면 dict.get(k) 를 쓴다.
    raise NotImplementedError('실습 TODO (8) : 정답 슬롯과 예측 슬롯을 비교해 틀린 슬롯 수 errors 를 계산하시오.')

    # SER 의 분모는 '정답(참조) 슬롯의 총 수'이다 (강의자료 9.3의 정의)
    return errors / len(gold_slots) if gold_slots else 0.0


try:
    # 강의자료 예시: TIME 슬롯만 틀림 → 1/3
    gold = {"PERSON": "Chris", "TIME": "10:30", "ROOM": "Gates 104"}
    pred = {"PERSON": "Chris", "TIME": "11:30", "ROOM": "Gates 104"}
    print("SER =", round(slot_error_rate(gold, pred), 3), "(기대: 0.333)")

    # 우리 챗봇의 슬롯 추출 성능 평가
    cases = [
        ("토요일 저녁 7시에 아바타 2명 예매해줘",
         {"movie": "아바타", "date": "토요일", "time": "저녁 7시", "count": "2명"}),
        ("내일 오후 3시 기생충 3장 부탁해",
         {"movie": "기생충", "date": "내일", "time": "오후 3시", "count": "3명"}),
        ("일요일 19시 30분 인터스텔라 1명",
         {"movie": "인터스텔라", "date": "일요일", "time": "19시 30분", "count": "1명"}),
    ]

    sers = []
    for utt, gold_slots in cases:
        pred_slots = extract_slots(utt)
        ser = slot_error_rate(gold_slots, pred_slots)
        sers.append(ser)
        print(f"{utt}\n  예측: {pred_slots}\n  SER = {ser:.3f}")

    print(f"\n평균 Slot Error Rate = {np.mean(sers):.3f}")
except NotImplementedError as e:
    sers = []
    print('▶ 미구현 :', e)


## 3-4. 생성 기반 챗봇과 멀티턴 문맥

- 기능대화(Task-oriented)와 달리 일상대화(Social conversation)는 정답이 하나가 아니다.
- 작은 생성 모델로 멀티턴 대화를 만들어 보고, 규칙 기반과 비교한다.
- (모델이 작아 응답 품질은 낮다. 여기서는 **문맥을 어떻게 누적하는지**를 보는 것이 목적이다)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

CHAT_MODEL = "microsoft/DialoGPT-small"      # 영어, 소형 대화 생성 모델
tok_chat = AutoTokenizer.from_pretrained(CHAT_MODEL)
model_chat = AutoModelForCausalLM.from_pretrained(CHAT_MODEL)

history_ids = None
for turn in ["Hello, how are you?", "What did you do yesterday?", "Do you like movies?"]:
    new_ids = tok_chat.encode(turn + tok_chat.eos_token, return_tensors="pt")
    # 이전 대화 이력에 이번 발화를 이어 붙인다 (= 대화 문맥 누적)
    input_ids = new_ids if history_ids is None else torch.cat([history_ids, new_ids], dim=-1)

    history_ids = model_chat.generate(
        input_ids, max_length=200, do_sample=True, top_p=0.9, temperature=0.8,
        pad_token_id=tok_chat.eos_token_id)

    reply = tok_chat.decode(history_ids[:, input_ids.shape[-1]:][0], skip_special_tokens=True)
    print("사용자:", turn)
    print("시스템:", reply, "\n")

> **토의**: 같은 발화를 여러 번 실행하면 매번 다른 응답이 나온다(`do_sample=True`).
> 기능대화라면 이런 성질이 왜 문제가 되는가? 일상대화에서는 왜 오히려 필요한가?

## 3-5. 규칙 기반과 생성 기반의 비교 정리

| 항목 | Frame 기반 (3-1) | 생성 기반 (3-4) |
|---|---|---|
| 응답의 예측 가능성 | 높음 (정해진 문장) | 낮음 (매번 달라짐) |
| 필요한 데이터 | 거의 없음 (규칙만) | 대량의 대화 데이터 |
| 도메인 확장 | 슬롯·규칙을 추가해야 함 | 학습 데이터에 의존 |
| 실패 방식 | 인식 실패(모름) | 그럴듯한 오답(환각) |
| 적합한 용도 | 예약·주문 등 기능대화 | 잡담·상담 등 일상대화 |

→ 실무에서는 두 방식을 결합하고, 근거가 필요한 답변은 **RAG**(9.1절)로 처리한다.

---
# 제출 과제

아래 3문항을 모두 수행한다. 코드 셀은 **실행 결과가 남은 상태로** 저장하고, 서술형 답안은 마크다운 셀에 작성한다.

- 제출 파일명 : `9주차_과제_학번_이름.ipynb`
- 본문의 **실습 TODO (1)~(8)** 이 모두 채워져 있어야 과제 셀이 정상 동작한다.
- 각 과제 아래에는 **채점 셀** 이 있다. 세 채점 셀이 모두 `통과!` 를 출력해야 정답으로 인정된다.
- 채점 셀은 수정하지 않는다. (수정하면 채점 결과가 무효 처리된다)


## 과제 1. 질의응답 시스템 확장과 실패 사례 분석 (30점)

**요구 사항**

1. `build_my_docs()` — `DOCS` 에 없던 **새 주제의 문서 3개 이상**을 반환한다. (각 문서 2문장 이상)
2. `build_my_eval()` — 추가한 문서에서 답할 수 있는 **(질문, 정답) 3쌍**을 반환한다.
   추출형 QA이므로 **정답 문자열이 문서 안에 그대로 들어 있어야** 한다.
3. `topk_table(eval_set, ks)` — `top_k` 를 1 / 3 / 5 로 바꾸어 가며 EM · F1 을 측정해
   `[(k, EM, F1)]` 을 반환한다. (0~100 스케일, `ks` 와 같은 순서)
4. **서술** : 검색이 실패해서 정답을 못 맞힌 사례와, 검색은 성공했지만 Reader가 실패한 사례를
   각각 하나씩 찾아 원인을 강의자료의 3단계(질문 처리 · 문서 처리 · 정답 처리) 용어로 설명한다.

> 문서를 추가하면 `DOC_MATRIX = vectorizer.fit_transform(DOCS)` 로 **문서 행렬을 반드시 다시 만들어야** 한다.
> (아래 셀에 이미 들어 있다)


In [ ]:
# ── 과제 1 ────────────────────────────────────────────────────────
def build_my_docs():
    """DOCS 에 없던 새 주제의 문서 3개 이상을 리스트로 반환한다. (각 문서는 2문장 이상)"""
    # TODO 1 : 기존 DOCS 와 겹치지 않는 새 주제의 문서를 3개 이상 직접 작성한다.
    #          각 문서는 마침표로 끝나는 문장을 2개 이상 포함해야 한다.
    raise NotImplementedError('과제 1-1 : 새 주제의 문서 3개 이상(각 2문장 이상)을 리스트로 반환하시오.')


def build_my_eval():
    """새로 추가한 문서에서 답을 찾을 수 있는 (질문, 정답) 3쌍을 반환한다."""
    # TODO 2 : 정답 문자열이 build_my_docs() 의 문서 안에 그대로 들어 있도록 질문 3개를 만든다.
    #          (추출형 QA 이므로 지문에 없는 표현은 정답이 될 수 없다)
    raise NotImplementedError('과제 1-2 : 새 문서에서 답할 수 있는 (질문, 정답) 3쌍을 리스트로 반환하시오.')


def topk_table(eval_set, ks=(1, 3, 5)):
    """top_k 를 바꾸어 가며 EM / F1 을 측정해 [(k, EM, F1)] 을 반환한다. (0~100 스케일)"""
    # TODO 3 : ks 의 각 k 에 대해
    #          open_domain_qa(q, top_k=k, verbose=False) 로 예측하고
    #          exact_match / token_f1 의 평균에 100 을 곱해 (k, EM, F1) 튜플로 모은다.
    #          결과는 ks 와 같은 순서로 담는다.
    raise NotImplementedError('과제 1-3 : ks 의 각 값에 대해 EM/F1 평균을 구해 [(k, EM, F1)] 로 반환하시오.')


try:
    MY_DOCS = build_my_docs()
    MY_EVAL = build_my_eval()

    # 문서를 추가했으면 벡터라이저와 문서 행렬을 반드시 다시 학습시켜야 한다.
    DOCS = DOCS + [d for d in MY_DOCS if d not in DOCS]
    DOC_MATRIX = vectorizer.fit_transform(DOCS)
    print("확장된 문서 수 :", len(DOCS), "\n")

    FULL_EVAL = EVAL_SET + MY_EVAL
    for k, em, f1 in topk_table(FULL_EVAL):
        print(f"top_k={k}: EM={em:5.1f}  F1={f1:5.1f}")
except NotImplementedError as e:
    MY_DOCS, MY_EVAL = [], []
    print('▶ 미구현 :', e)


In [ ]:
# [과제 1] 채점 — 기대 조건이 하드코딩되어 있다.
try:
    # ── 1-1 : 새 문서 ────────────────────────────────────────────
    _docs = build_my_docs()
    assert isinstance(_docs, list), '1-1 은 리스트를 반환해야 합니다.'
    assert len(_docs) >= 3, f'문서를 3개 이상 추가해야 합니다. (현재 {len(_docs)}개)'
    assert all(isinstance(d, str) for d in _docs), '문서는 문자열이어야 합니다.'
    assert len(set(_docs)) == len(_docs), '같은 문서를 반복해서 넣을 수 없습니다.'
    assert all(len(d.strip()) >= 40 for d in _docs), '각 문서는 40자 이상이어야 합니다. (빈 문자열·자리표시자 금지)'
    for _d in _docs:
        _sents = [s for s in re.split(r'[.!?]', _d) if s.strip()]
        assert len(_sents) >= 2, f'각 문서는 2문장 이상이어야 합니다. (문제 문서: {_d[:20]}...)'
    _ORIG_HEAD = {'마하트마 간디는 1869년', '1930년 간디는 영국의 ', '자와할랄 네루는 인도의 초',
                  '세종대왕은 1443년에 훈', '트랜스포머는 2017년에 ', 'BLEU는 기계 번역문과 ',
                  '대화 시스템은 음성 인식,'}
    assert not any(_d[:14] in _ORIG_HEAD for _d in _docs), '기존 DOCS 를 그대로 복사하면 안 됩니다.'

    # ── 1-2 : 새 평가 질문 ───────────────────────────────────────
    _ev = build_my_eval()
    assert isinstance(_ev, list) and len(_ev) == 3, f'(질문, 정답) 3쌍이어야 합니다. (현재 {len(_ev)}쌍)'
    assert all(isinstance(x, (tuple, list)) and len(x) == 2 for x in _ev), '(질문, 정답) 쌍이어야 합니다.'
    assert all(isinstance(v, str) and v.strip() for x in _ev for v in x), '질문과 정답은 비어 있지 않은 문자열이어야 합니다.'
    assert len({q for q, _ in _ev}) == 3, '질문이 중복되었습니다.'
    assert all(q.strip().endswith('?') for q, _ in _ev), '질문은 물음표로 끝나야 합니다.'
    _joined = ' '.join(_docs)
    for _q, _g in _ev:
        assert _g in _joined, f'정답 "{_g}" 이 추가한 문서 안에 문자열 그대로 존재해야 합니다. (추출형 QA)'

    # ── 1-3 : top_k 별 EM/F1 ────────────────────────────────────
    #   정답이 절대 존재하지 않는 평가셋을 넣어 본다. 상수를 돌려주는 답안은 여기서 걸린다.
    _bogus = [(q, 'ZZZ존재하지않는정답ZZZ') for q, _ in EVAL_SET[:2]]
    _tb = topk_table(_bogus, ks=(1,))
    assert isinstance(_tb, list) and len(_tb) == 1, 'ks 의 길이와 결과 행 수가 같아야 합니다.'
    assert len(_tb[0]) == 3, '각 행은 (k, EM, F1) 3-튜플이어야 합니다.'
    assert _tb[0][0] == 1, '각 행의 첫 원소는 k 여야 합니다.'
    assert _tb[0][1] == 0.0 and _tb[0][2] == 0.0, \
        '정답이 문서에 전혀 없는 평가셋인데 EM/F1 이 0 이 아닙니다. (상수를 반환하고 있지 않은지 확인하세요)'

    _tr = topk_table(EVAL_SET[:3], ks=(1, 3))
    assert [r[0] for r in _tr] == [1, 3], 'ks 의 순서대로 결과를 담아야 합니다.'
    assert all(0.0 <= r[1] <= 100.0 and 0.0 <= r[2] <= 100.0 for r in _tr), 'EM·F1 은 0~100 스케일이어야 합니다.'
    assert all(r[2] + 1e-9 >= r[1] for r in _tr), 'F1 은 항상 EM 이상이어야 합니다. (평균 계산식을 확인하세요)'
    assert max(r[2] for r in _tr) > 0.0, '정상 평가셋인데 F1 이 모두 0 입니다. 파이프라인이 동작하지 않습니다.'
    assert (_tr[0][1], _tr[0][2]) != (_tb[0][1], _tb[0][2]), \
        '서로 다른 평가셋인데 결과가 같습니다. eval_set 을 실제로 사용하고 있는지 확인하세요.'

    print(f'통과!  추가 문서 {len(_docs)}개, 새 질문 {len(_ev)}개, top_k 표 =',
          [(k, round(em, 1), round(f1, 1)) for k, em, f1 in _tr])
except NotImplementedError as e:
    print('▶ 미구현 :', e)


### 과제 1 서술형 답안

- 검색(문서 처리) 실패 사례:
- Reader(정답 처리) 실패 사례:
- 원인 분석:

## 과제 2. 번역 품질과 디코딩 전략 분석 (40점)

**요구 사항**

1. `build_my_pairs()` — **(한국어 문장, 영어 참조 번역) 5쌍**을 반환한다.
   6어절 이하의 짧은 문장 2개 이상, 10어절 이상의 긴 문장 2개 이상을 포함할 것.
   (참조 번역은 본인이 작성하거나 신뢰할 수 있는 번역을 사용한다)
2. `beam_bleu_table(pairs, beam_list)` — `num_beams` 를 1 / 2 / 4 / 8 로 바꾸어
   **평균 BLEU** 를 `[(beams, my_bleu 평균, sacrebleu 평균)]` 로 반환한다.
   (직접 구현한 `my_bleu` 와 `sacrebleu` 를 **모두** 사용해 비교할 것)
3. `length_bleu(pairs)` — 문장 길이(어절 수)에 따라 BLEU가 어떻게 달라지는지
   `[(어절 수, BLEU)]` 로 반환한다.
4. **서술** : BLEU 점수는 낮지만 사람이 보기에 좋은 번역인 사례를 하나 제시하고,
   강의자료에서 다룬 BLEU의 한계와 연결하여 설명한다. 또한 어떤 지표로 보완할 수 있는지 쓴다.

> BLEU 는 대소문자에 민감하다. 가설과 참조를 **모두 소문자로 맞춘 뒤** 계산할 것.


In [ ]:
# ── 과제 2 ────────────────────────────────────────────────────────
def build_my_pairs():
    """(한국어 문장, 영어 참조 번역) 5쌍을 반환한다."""
    # TODO 1 : 서로 다른 한국어 문장 5개와 각각의 영어 참조 번역을 직접 작성한다.
    #          - 6어절 이하의 짧은 문장을 2개 이상 포함할 것
    #          - 10어절 이상의 긴 문장을 2개 이상 포함할 것
    #          - 참조 번역에는 한글을 넣지 않는다.
    raise NotImplementedError('과제 2-1 : (한국어 문장, 영어 참조 번역) 5쌍을 리스트로 반환하시오.')


def beam_bleu_table(pairs, beam_list=(1, 2, 4, 8)):
    """빔 크기별 평균 BLEU 를 [(beams, my_bleu 평균, sacrebleu 평균)] 로 반환한다. (0~100 스케일)"""
    # TODO 2 : beam_list 의 각 beams 에 대해
    #          translate(src, tok_ko2en, model_ko2en, num_beams=beams) 로 번역하고
    #          my_bleu 와 sacrebleu.sentence_bleu 의 평균을 각각 구해 튜플로 모은다.
    #          - 대소문자 차이로 n-gram 이 어긋나지 않도록 가설과 참조를 모두 소문자로 맞춘다.
    #          - my_bleu 의 반환값은 0~1 이므로 100 을 곱하고, sacrebleu 의 .score 는 이미 0~100 이다.
    raise NotImplementedError('과제 2-2 : 빔 크기별 평균 BLEU 를 [(beams, my_bleu 평균, sacrebleu 평균)] 로 반환하시오.')


def length_bleu(pairs, num_beams=4):
    """각 문장의 (원문 어절 수, BLEU) 를 리스트로 반환한다. (0~100 스케일)"""
    # TODO 3 : 각 쌍을 num_beams 로 번역해 my_bleu 로 점수를 구하고,
    #          원문의 어절 수(len(src.split())) 와 함께 (어절 수, BLEU) 로 모은다.
    raise NotImplementedError('과제 2-3 : 각 문장의 (원문 어절 수, BLEU) 를 리스트로 반환하시오.')


try:
    MY_PAIRS = build_my_pairs()
    print("[빔 크기별 평균 BLEU]")
    for beams, mine, sacre in beam_bleu_table(MY_PAIRS):
        print(f"  num_beams={beams}: my_bleu={mine:6.2f}   sacrebleu={sacre:6.2f}")

    print("\n[문장 길이와 BLEU]")
    for n_words, b in sorted(length_bleu(MY_PAIRS)):
        print(f"  원문 어절 수 {n_words:2d} → BLEU {b:6.2f}")
except NotImplementedError as e:
    MY_PAIRS = []
    print('▶ 미구현 :', e)


In [ ]:
# [과제 2] 채점 — 기대 조건이 하드코딩되어 있다.
try:
    # ── 2-1 : 문장쌍 ────────────────────────────────────────────
    _p = build_my_pairs()
    assert isinstance(_p, list) and len(_p) == 5, f'문장쌍은 정확히 5개여야 합니다. (현재 {len(_p)}개)'
    assert all(isinstance(x, (tuple, list)) and len(x) == 2 for x in _p), '(한국어, 영어) 쌍이어야 합니다.'
    _ko = [k for k, _ in _p]
    _en = [e for _, e in _p]
    assert all(isinstance(s, str) and s.strip() for s in _ko + _en), '빈 문자열은 넣을 수 없습니다.'
    assert len(set(_ko)) == 5, '같은 한국어 문장을 반복해서 넣을 수 없습니다.'
    assert all(any('가' <= ch <= '힣' for ch in k) for k in _ko), '첫 번째 원소는 한국어 문장이어야 합니다.'
    assert not any(any('가' <= ch <= '힣' for ch in e) for e in _en), '두 번째 원소는 영어 참조 번역이어야 합니다.'
    assert all(len(e.split()) >= 3 for e in _en), '참조 번역이 너무 짧습니다. (3어절 이상)'
    assert sum(len(k.split()) <= 6 for k in _ko) >= 2, '6어절 이하의 짧은 문장이 2개 이상 필요합니다.'
    assert sum(len(k.split()) >= 10 for k in _ko) >= 2, '10어절 이상의 긴 문장이 2개 이상 필요합니다.'

    # ── 2-2 : 빔 크기별 표 ──────────────────────────────────────
    _t = beam_bleu_table(_p[:2])
    assert isinstance(_t, list), '2-2 는 리스트를 반환해야 합니다.'
    assert [r[0] for r in _t] == [1, 2, 4, 8], 'beam_list 의 순서대로 (beams, my, sacre) 를 담아야 합니다.'
    assert all(len(r) == 3 for r in _t), '각 행은 (beams, my_bleu 평균, sacrebleu 평균) 3-튜플이어야 합니다.'
    assert all(0.0 <= r[1] <= 100.0 and 0.0 <= r[2] <= 100.0 for r in _t), 'BLEU 는 0~100 스케일이어야 합니다.'

    #   참조 번역을 실제로 쓰고 있는지 확인한다. 상수를 돌려주는 답안은 여기서 걸린다.
    _good = beam_bleu_table(_p[:1], beam_list=(4,))[0]
    _bad = beam_bleu_table([(_p[0][0], 'zzz qqq xxx yyy www')], beam_list=(4,))[0]
    assert (_good[1], _good[2]) != (_bad[1], _bad[2]), \
        '참조 번역을 바꿨는데 점수가 그대로입니다. pairs 의 참조 번역을 실제로 사용하고 있는지 확인하세요.'
    assert _bad[1] < 5.0 and _bad[2] < 5.0, \
        '전혀 무관한 참조 번역인데 BLEU 가 너무 높습니다. n-gram 일치 계산을 확인하세요.'

    # ── 2-3 : 길이와 BLEU ──────────────────────────────────────
    _lb = length_bleu(_p)
    assert isinstance(_lb, list) and len(_lb) == 5, f'문장 5개에 대한 결과여야 합니다. (현재 {len(_lb)}개)'
    assert all(isinstance(x, (tuple, list)) and len(x) == 2 for x in _lb), '(어절 수, BLEU) 튜플이어야 합니다.'
    assert sorted(int(n) for n, _ in _lb) == sorted(len(k.split()) for k in _ko), \
        '첫 원소가 원문의 어절 수와 맞지 않습니다.'
    assert all(0.0 <= b <= 100.0 for _, b in _lb), 'BLEU 는 0~100 스케일이어야 합니다.'
    assert len({round(float(b), 6) for _, b in _lb}) > 1, \
        '다섯 문장의 BLEU 가 모두 같습니다. 상수를 반환하고 있지 않은지 확인하세요.'

    print('통과!  빔 크기별 평균 BLEU :', [(r[0], round(r[1], 2), round(r[2], 2)) for r in _t])
except NotImplementedError as e:
    print('▶ 미구현 :', e)


### 과제 2 서술형 답안

- BLEU는 낮지만 좋은 번역인 사례:
- 원인 (BLEU의 한계와 연결):
- 보완 지표와 이유:

## 과제 3. 대화 시스템 도메인 확장 (30점)

**요구 사항**

1. `build_frame_v2()` — 영화 예매 프레임(`FRAME`)과 **별도의 새 도메인 프레임**을 반환한다.
   (예: 식당 예약 = 메뉴 / 인원 / 날짜 / 시간, 항공 예약 = 출발지 / 도착지 / 날짜 / 좌석)
   슬롯 3개 이상, 각 슬롯은 `{"질문": ..., "값": None, "우선순위": 정수}` 구조, 우선순위는 서로 달라야 한다.
2. `extract_slots_v2(utterance)` — 새 도메인의 슬롯 추출기를 작성한다. (정규식 또는 사전 매칭)
   찾지 못한 슬롯은 키를 넣지 않고, 아무것도 못 찾으면 빈 딕셔너리를 반환한다.
3. `dialogue_manager(utterances)` — 3-2의 의도 분류기로 **의도에 따라 프레임을 선택**하여
   슬롯을 채우고 `(의도, 프레임)` 을 반환한다.
4. `build_my_cases()` — 새 도메인 발화 **5개**와 정답 슬롯을 만들고, **평균 Slot Error Rate** 가
   **0.3 이하**가 되도록 규칙을 다듬는다.
5. **서술** : 실패한 발화 사례를 제시하고, 규칙 기반 방식의 어떤 한계 때문인지 설명한다.
   또한 데이터 기반(신경망) 방식으로 바꾸면 무엇이 나아지고 무엇이 나빠지는지 쓴다.


In [ ]:
# ── 과제 3 ────────────────────────────────────────────────────────
def build_frame_v2():
    """새 도메인(예: 식당 예약, 항공 예약)의 프레임을 FRAME 과 같은 구조로 반환한다."""
    # TODO 1 : 슬롯을 3개 이상 설계한다. 각 슬롯의 값은
    #          {"질문": "...", "값": None, "우선순위": 정수} 구조여야 하고,
    #          우선순위는 서로 겹치지 않아야 한다.
    #          슬롯 이름 집합은 영화 예매 프레임(FRAME)과 달라야 한다.
    raise NotImplementedError('과제 3-1 : 새 도메인 프레임을 {슬롯: {"질문", "값", "우선순위"}} 구조로 반환하시오.')


def extract_slots_v2(utterance):
    """새 도메인 발화에서 슬롯 값을 추출해 딕셔너리로 반환한다."""
    # TODO 2 : 정규식 또는 사전 매칭으로 슬롯을 추출한다.
    #          - 찾지 못한 슬롯은 키 자체를 넣지 않는다.
    #          - 아무것도 찾지 못하면 빈 딕셔너리를 반환한다.
    #          - build_frame_v2() 에 없는 슬롯 이름을 만들면 안 된다.
    raise NotImplementedError('과제 3-2 : 새 도메인 발화에서 슬롯을 추출해 딕셔너리로 반환하시오.')


def dialogue_manager(utterances):
    """의도 분류 → 프레임 선택 → 슬롯 채우기 → 응답 출력. (의도, 채워진 프레임) 을 반환한다."""
    # TODO 3 : 첫 발화의 의도를 clf.predict(intent_vec.transform([utt]))[0] 로 구한다.
    #          의도가 'movie_booking' 이면 FRAME 과 extract_slots 를,
    #          그렇지 않으면 build_frame_v2() 와 extract_slots_v2 를 사용한다.
    #          발화를 순서대로 처리하며 빈 슬롯이 있으면 그 슬롯의 질문을,
    #          모두 채워졌으면 확정 메시지를 출력하고 (의도, 프레임) 을 반환한다.
    raise NotImplementedError('과제 3-3 : 의도에 따라 프레임을 고르는 dialogue_manager 를 구현해 (의도, 프레임) 을 반환하시오.')


def build_my_cases():
    """새 도메인 발화 5개와 정답 슬롯을 [(발화, {슬롯: 값})] 로 반환한다."""
    # TODO 4 : 새 도메인 발화 5개를 직접 만들고, 각 발화의 정답 슬롯을 손으로 적는다.
    #          정답 슬롯의 키는 build_frame_v2() 의 슬롯 이름이어야 한다.
    raise NotImplementedError('과제 3-4 : 새 도메인 발화 5개와 정답 슬롯을 [(발화, {슬롯: 값})] 로 반환하시오.')


try:
    FRAME_V2 = build_frame_v2()
    MY_CASES = build_my_cases()

    _sers = []
    for _utt, _gold in MY_CASES:
        _pred = extract_slots_v2(_utt)
        _s = slot_error_rate(_gold, _pred)
        _sers.append(_s)
        print(f"{_utt}\n  예측: {_pred}\n  SER = {_s:.3f}")
    print(f"\n새 도메인 평균 Slot Error Rate = {np.mean(_sers):.3f}\n")

    print("[대화 관리자 시연]")
    dialogue_manager(["영화 예매하고 싶어요", "아바타 보려고요", "토요일 저녁 7시 2명이요"])
    print()
    dialogue_manager([MY_CASES[0][0]])
except NotImplementedError as e:
    FRAME_V2, MY_CASES = {}, []
    print('▶ 미구현 :', e)


In [ ]:
# [과제 3] 채점 — 기대 조건이 하드코딩되어 있다.
try:
    # ── 3-1 : 새 프레임 ─────────────────────────────────────────
    _fr = build_frame_v2()
    assert isinstance(_fr, dict), '3-1 은 딕셔너리를 반환해야 합니다.'
    assert len(_fr) >= 3, f'새 프레임의 슬롯은 3개 이상이어야 합니다. (현재 {len(_fr)}개)'
    for _k, _v in _fr.items():
        assert isinstance(_v, dict) and {'질문', '값', '우선순위'} <= set(_v), \
            f'슬롯 "{_k}" 에 질문/값/우선순위 키가 모두 있어야 합니다.'
        assert isinstance(_v['질문'], str) and _v['질문'].strip(), f'슬롯 "{_k}" 의 질문이 비어 있습니다.'
        assert _v['값'] is None, '초기 프레임의 값은 모두 None 이어야 합니다.'
        assert isinstance(_v['우선순위'], int), '우선순위는 정수여야 합니다.'
    assert len({_v['우선순위'] for _v in _fr.values()}) == len(_fr), '우선순위가 서로 겹칩니다.'
    assert set(_fr) != set(FRAME), '영화 예매 프레임과 다른 새 도메인이어야 합니다.'

    # ── 3-4 : 평가 발화 ─────────────────────────────────────────
    _cases = build_my_cases()
    assert isinstance(_cases, list) and len(_cases) == 5, f'평가 발화는 정확히 5개여야 합니다. (현재 {len(_cases)}개)'
    assert all(isinstance(x, (tuple, list)) and len(x) == 2 for x in _cases), '(발화, 정답 슬롯) 쌍이어야 합니다.'
    assert len({u for u, _ in _cases}) == 5, '같은 발화를 반복해서 넣을 수 없습니다.'
    for _u, _g in _cases:
        assert isinstance(_u, str) and len(_u.strip()) >= 5, f'발화가 너무 짧습니다: "{_u}"'
        assert isinstance(_g, dict) and _g, f'"{_u}" 의 정답 슬롯이 비어 있습니다.'
        assert set(_g) <= set(_fr), f'정답 슬롯 키 {sorted(set(_g) - set(_fr))} 가 프레임에 없습니다.'

    # ── 3-2 : 슬롯 추출 ─────────────────────────────────────────
    assert extract_slots_v2('') == {}, \
        '정보가 전혀 없는 발화에는 빈 딕셔너리를 반환해야 합니다. (고정된 값을 돌려주면 안 됩니다)'
    _preds = [extract_slots_v2(u) for u, _ in _cases]
    assert all(isinstance(p, dict) for p in _preds), 'extract_slots_v2 는 딕셔너리를 반환해야 합니다.'
    assert all(set(p) <= set(_fr) for p in _preds), '프레임에 없는 슬롯 이름을 만들면 안 됩니다.'
    assert len({tuple(sorted(p.items())) for p in _preds}) >= 3, \
        '서로 다른 발화인데 추출 결과가 거의 같습니다. 발화 내용을 실제로 분석해야 합니다.'
    _ser = float(np.mean([slot_error_rate(_g, _p) for (_, _g), _p in zip(_cases, _preds)]))
    assert _ser <= 0.3, f'평균 Slot Error Rate 가 {_ser:.3f} 로 너무 높습니다. (0.3 이하가 되도록 규칙을 보완하세요)'

    # ── 3-3 : 대화 관리자 ───────────────────────────────────────
    _out = dialogue_manager(['영화 예매하고 싶어요', '아바타 보려고요'])
    assert isinstance(_out, (tuple, list)) and len(_out) == 2, '(의도, 프레임) 을 반환해야 합니다.'
    _intent, _frame = _out
    assert isinstance(_intent, str) and isinstance(_frame, dict), '(의도 문자열, 프레임 딕셔너리) 여야 합니다.'
    assert _intent == 'movie_booking', f'영화 예매 발화의 의도는 movie_booking 이어야 하는데 {_intent} 입니다.'
    assert _frame.get('movie', {}).get('값') == '아바타', \
        "영화 예매 의도일 때는 FRAME 을 골라 movie 슬롯('아바타')을 채워야 합니다."
    _out2 = dialogue_manager([_cases[0][0]])
    assert isinstance(_out2, (tuple, list)) and len(_out2) == 2, '(의도, 프레임) 을 반환해야 합니다.'

    print(f'통과!  새 도메인 슬롯 {sorted(_fr)}, 평균 SER = {_ser:.3f}, 의도 = {_intent}')
except NotImplementedError as e:
    print('▶ 미구현 :', e)


### 과제 3 서술형 답안

- 새 도메인 설계 의도:
- 실패 사례와 원인:
- 데이터 기반 방식으로 전환 시 장점 / 단점:

---
## 제출 체크리스트

- [ ] 실습 TODO (1)~(8) 모두 작성 및 실행 완료
- [ ] Part 1~3의 모든 셀이 오류 없이 실행된 상태로 저장
- [ ] 과제 1~3의 코드와 서술형 답안 작성 완료
- [ ] **세 개의 채점 셀이 모두 `통과!` 를 출력**
- [ ] 파일명 : `9주차_과제_학번_이름.ipynb`

### 자주 발생하는 문제

| 증상 | 해결 |
|---|---|
| `TypeError: only length-1 arrays can be converted...` (retrieve) | `sims` 가 2차원이다. `[0]` 또는 `.ravel()` 로 1차원으로 만들 것 |
| `ImportError: cannot import name ... from huggingface_hub` | `[0-1]` 셀을 다시 실행한 뒤 **런타임 재시작** → `[0-2]` 셀부터 이어서 실행 |
| 모델 다운로드가 매우 느림 | Colab 런타임을 GPU로 변경하면 전체 실행 시간이 크게 줄어든다 |
| `pipeline(... device=DEVICE)` 인자 오류 | `transformers` 버전 차이. `device=DEVICE` 를 제거하고 실행 |
| 그래프에서 한글이 □ 로 보임 | `[0-1]` 실행 후 **런타임 재시작** → `[0-2]` 셀 다시 실행 |

---

**수고하셨습니다.** 다음 주차에서는 토픽 모델링 · 감성 분석 · 기계 독해를 다룹니다.
